# 3.2 · 资源（Resources）与资源模板

**Resources and resource templates**

> **学完本课你能：** 将数据以 URI 寻址的资源和 RFC 6570 模板形式暴露出来，携带正确的元数据提供文本与二进制内容，并正确处理“未找到”、注解和变更通知。

官方阅读材料：[Spec: Resources](https://modelcontextprotocol.io/specification/2026-07-28/server/resources) · [Understanding MCP servers](https://modelcontextprotocol.io/docs/2026-07-28/learn/server-concepts)

从上到下依次运行单元格（`Shift+Enter`），然后改动代码再重新运行。所有内容都在本地运行：服务器位于当前内核、临时目录或 localhost 上。调用真实大语言模型（LLM）的单元格只有在设置了 `ANTHROPIC_API_KEY` 时才会执行，否则自动跳过。

### 1. 一个带直接资源和模板的服务器
`MCPServer` 根据 URI 来判断：不含 `{...}` 的是直接资源（Direct Resource，出现在 `resources/list` 中），只要含有任何变量就是模板（Resource Template，出现在 `resources/templates/list` 中）。返回 `str` 得到 `text`，返回 `bytes` 得到 base64 编码的 `blob`。我们还用 `FileResource` 以 `file://` URI 注册了一个真实文件。

In [ ]:
import base64
import json
import logging
import struct
import tempfile
import zlib
from pathlib import Path

import mcp_types as types
from mcp import Client
from mcp.server import MCPServer
from mcp.server.mcpserver.resources.types import FileResource
from mcp.shared.exceptions import MCPError

logging.getLogger("mcp").setLevel(logging.CRITICAL)


def show(model, limit=1500):
    data = model.model_dump(by_alias=True, exclude_none=True, mode="json")
    data.pop("_meta", None)
    print(json.dumps(data, indent=2, ensure_ascii=False)[:limit])


def tiny_png() -> bytes:
    chunk = lambda kind, body: (struct.pack(">I", len(body)) + kind + body
                                + struct.pack(">I", zlib.crc32(kind + body) & 0xFFFFFFFF))
    return (b"\x89PNG\r\n\x1a\n" + chunk(b"IHDR", struct.pack(">IIBBBBB", 1, 1, 8, 2, 0, 0, 0))
            + chunk(b"IDAT", zlib.compress(b"\x00\xff\x00\x00")) + chunk(b"IEND", b""))


project = Path(tempfile.mkdtemp())
(project / "README.md").write_text("# Trip planner\nPlan trips with MCP.\n")

server = MCPServer("travel-data")
server.add_resource(FileResource(uri=(project / "README.md").as_uri(), path=project / "README.md",
                                 name="README.md", title="Project README", mime_type="text/markdown"))


@server.resource("calendar://events/2026", title="2026 calendar", mime_type="application/json",
                 annotations=types.Annotations(audience=["assistant"], priority=0.9,
                                               last_modified="2026-07-28T09:00:00Z"))
def calendar() -> str:
    return json.dumps({"busy": ["2026-08-03", "2026-08-04"]})


@server.resource("travel://preferences", title="Travel preferences",
                 annotations=types.Annotations(audience=["user", "assistant"], priority=0.6))
def preferences() -> str:
    return "Window seat. Prefers boutique hotels. Vegetarian."


@server.resource("images://logo.png", title="Company logo", mime_type="image/png",
                 annotations=types.Annotations(audience=["user"], priority=0.1))
def logo() -> bytes:
    return tiny_png()


@server.resource("weather://forecast/{city}/{date}", title="Weather Forecast", mime_type="application/json",
                 description="Get weather forecast for any city and date")
def forecast(city: str, date: str) -> str:
    return json.dumps({"city": city, "date": date, "high_c": 29, "conditions": "sunny"})


@server.resource("travel://activities/{city}/{category}", title="Activities", mime_type="application/json")
def activities(city: str, category: str) -> str:
    catalogue = {"museums": ["Picasso Museum", "MNAC"], "beaches": ["Barceloneta", "Bogatell"]}
    return json.dumps({"city": city, "category": category, "items": catalogue.get(category, [])})


DOCS = {"guides/packing.md": "Bring an adapter.", "guides/visas.md": "Check visa rules early."}


@server.resource("docs://{+path}", title="Travel guides", mime_type="text/markdown")
def docs(path: str) -> str:
    return DOCS[path]


print("defined:", server.name)

### 2. 发现：直接资源与模板
这是两个不同的列举调用。留意其中的注解（Annotations），以及模板携带的是 `uriTemplate` 而不是 `uri`。

In [ ]:
async with Client(server) as client:
    print("capabilities.resources:", client.server_capabilities.resources)
    direct = await client.list_resources()
    print("\nresources/list:")
    for r in direct.resources:
        a = r.annotations.model_dump(by_alias=True, exclude_none=True) if r.annotations else {}
        print(f"  {r.uri:<44} {r.mime_type:<18} {a}")
    templates = await client.list_resource_templates()
    print("\nresources/templates/list:")
    for t in templates.resource_templates:
        print(f"  {t.uri_template:<44} {t.mime_type}")

### 3. 读取文本、二进制和模板化资源
模板（Resource Template）在<strong>客户端</strong>按 RFC 6570 规则展开（这里用的是 SDK 的 `UriTemplate`；注意空格和 `ã` 被百分号编码了），然后用普通的 `resources/read` 读取。

In [ ]:
from mcp.shared.uri_template import UriTemplate

async with Client(server) as client:
    show(await client.read_resource("calendar://events/2026"))

    blob = (await client.read_resource("images://logo.png")).contents[0]
    raw = base64.b64decode(blob.blob)
    png_ok = raw.startswith(bytes([0x89]) + b"PNG")
    print(f"\nblob: mimeType={blob.mime_type}, {len(raw)} bytes after base64 decoding, PNG signature ok: {png_ok}")

    uri = UriTemplate.parse("weather://forecast/{city}/{date}").expand({"city": "São Paulo", "date": "2026-08-01"})
    print("\nexpanded:", uri)
    print("contents:", (await client.read_resource(uri)).contents[0].text)

    uri = UriTemplate.parse("docs://{+path}").expand({"path": "guides/packing.md"})
    print("reserved expansion keeps slashes:", uri, "->", (await client.read_resource(uri)).contents[0].text)

### 4. 未找到是 `-32602`——而且绝不是空数组
先读取一个什么都匹配不上的 URI，再读取一个模板能匹配但数据缺失的 URI。然后通过一个 stdio 子进程，在线路上以原始 JSON 的形式查看同样的错误。

In [ ]:
async with Client(server) as client:
    for uri in ["calendar://events/1999", "docs://guides/nonexistent.md"]:
        try:
            await client.read_resource(uri)
        except MCPError as err:
            print(f"{uri:<32} -> code={err.code} message={err.message!r} data={err.data}")

第二种情况暴露了一个 SDK 细节：模板处理函数抛出异常（这里是 `KeyError`）时，会变成 `-32603` <em>内部错误</em>，而不是未找到。要遵循规范，就应在处理函数中抛出 SDK 的 `ResourceNotFoundError`：

In [ ]:
from mcp.server.mcpserver.exceptions import ResourceNotFoundError


@server.resource("guides://{+path}", mime_type="text/markdown")
def guides(path: str) -> str:
    if path not in DOCS:
        raise ResourceNotFoundError(f"Resource not found: guides://{path}")
    return DOCS[path]


async with Client(server) as client:
    try:
        await client.read_resource("guides://nope.md")
    except MCPError as err:
        print("fixed:", err.code, err.message, err.data)


def is_not_found(err: MCPError) -> bool:
    """Client-side: accept the 2026-07-28 code and the legacy one."""
    return err.code in (-32602, -32002)


print("treat as not found:", is_not_found(MCPError(-32002, "Resource not found")))

In [ ]:
import subprocess
import sys
import textwrap

workdir = Path(tempfile.mkdtemp())
(workdir / "res_server.py").write_text(textwrap.dedent('''
    from mcp.server import MCPServer
    server = MCPServer("res")

    @server.resource("config://app", mime_type="application/json")
    def config() -> str:
        return '{"theme": "dark"}'

    if __name__ == "__main__":
        server.run(transport="stdio")
'''))
proc = subprocess.Popen([sys.executable, str(workdir / "res_server.py")], stdin=subprocess.PIPE,
                        stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)
META = {"io.modelcontextprotocol/protocolVersion": "2026-07-28",
        "io.modelcontextprotocol/clientCapabilities": {},
        "io.modelcontextprotocol/clientInfo": {"name": "by-hand", "version": "0.1"}}
for i, uri in enumerate(["config://app", "config://missing"], 1):
    msg = {"jsonrpc": "2.0", "id": i, "method": "resources/read", "params": {"uri": uri, "_meta": META}}
    proc.stdin.write(json.dumps(msg) + "\n")
    proc.stdin.flush()
    reply = json.loads(proc.stdout.readline())
    reply.get("result", {}).pop("_meta", None)
    print(">>>", json.dumps({k: v for k, v in msg["params"].items() if k != "_meta"}))
    print("<<<", json.dumps(reply), "\n")
proc.stdin.close()
_ = proc.wait(timeout=10)

### 5. 动手破坏：目录穿越
规范规定服务器必须（MUST）净化路径。SDK 的模板默认会拒绝提取出的参数中含有 `..` 组件或绝对路径的情况（`ResourceSecurity`），因此这类请求根本到不了我们的处理函数。

In [ ]:
async with Client(server) as client:
    for uri in ["docs://guides/visas.md", "docs://../../etc/passwd", "docs:///etc/passwd", "docs://guides/%2e%2e/%2e%2e/secret"]:
        try:
            text = (await client.read_resource(uri)).contents[0].text
            print(f"{uri:<38} OK   {text}")
        except MCPError as err:
            print(f"{uri:<38} {err.code} {err.message}")

如果你自己拼接路径（自定义的底层处理函数、`FileResource` 工厂），也要做同样的检查：先解析路径，再验证它仍位于根目录之内。

In [ ]:
def safe_join(root: Path, relative: str) -> Path:
    candidate = (root / relative).resolve()
    if not candidate.is_relative_to(root.resolve()):
        raise PermissionError(f"path escapes root: {relative}")
    return candidate


for rel in ["README.md", "../../etc/passwd"]:
    try:
        print(rel, "->", safe_join(project, rel))
    except PermissionError as exc:
        print(rel, "->", exc)

### 6. 用注解挑选上下文的宿主
资源（Resource）是应用驱动的：由宿主（Host）来选择。这个小小的选择器只保留面向助手的条目，按优先级排序，并在达到字符预算时停止。

In [ ]:
async def select_context(client, budget_chars=120):
    resources = (await client.list_resources()).resources
    for_model = [r for r in resources
                 if not r.annotations or not r.annotations.audience or "assistant" in r.annotations.audience]
    ranked = sorted(for_model, key=lambda r: (r.annotations.priority if r.annotations and r.annotations.priority is not None else 0.5),
                    reverse=True)
    chosen, used = [], 0
    for r in ranked:
        item = (await client.read_resource(str(r.uri))).contents[0]
        text = getattr(item, "text", None)
        if text is None or used + len(text) > budget_chars:
            print(f"skip   {r.uri} ({'binary' if text is None else 'over budget'})")
            continue
        chosen.append((str(r.uri), text))
        used += len(text)
        print(f"attach {r.uri} ({len(text)} chars)")
    return chosen


async with Client(server) as client:
    context = await select_context(client)
print("\nprompt context:\n" + "\n".join(f"<resource uri='{u}'>{t.strip()}</resource>" for u, t in context))

徽标（`audience: ["user"]`）永远不会被考虑交给模型，日历（priority 0.9）排在最前。

### 7. 预览：监听资源变化
在 2026-07-28 版本中，客户端（Client）通过 `subscriptions/listen` 监听 URI（第 3.4 课有详细介绍）。一个工具修改了日历并发布 `notifications/resources/updated`；该通知（Notification）只写明 URI，因此客户端会重新读取。

In [ ]:
from mcp.server.mcpserver import Context

busy_days = ["2026-08-03", "2026-08-04"]


@server.resource("calendar://busy", mime_type="application/json")
def busy() -> str:
    return json.dumps(busy_days)


@server.tool()
async def block_day(day: str, ctx: Context) -> str:
    """Mark a day as busy in the calendar."""
    busy_days.append(day)
    await ctx.notify_resource_updated("calendar://busy")
    return f"blocked {day}"


async with Client(server) as client:
    print("before:", (await client.read_resource("calendar://busy")).contents[0].text)
    async with client.listen(resource_subscriptions=["calendar://busy"]) as sub:
        await client.call_tool("block_day", {"day": "2026-08-10"})
        event = await anext(sub)
        print("event:", event)
        print("after: ", (await client.read_resource(event.uri)).contents[0].text)

### 动手练习
1. 添加一个模板 `travel://flights/{origin}/{destination}`，以及一个 `audience: ["user", "assistant"]`、`priority: 1.0` 的直接资源（Direct Resource）。重新运行第 2 节和第 6 节：新资源在选择结果中排在哪里？
2. 添加一个使用查询展开的模板 `search://hotels{?city,max_price}`（处理函数的参数需要默认值）。用 `UriTemplate` 针对两组不同的输入展开它，并读取这两个结果。
3. 让 `forecast` 对不符合 `YYYY-MM-DD` 格式的日期抛出 `ResourceNotFoundError`，然后思考：“格式错误”真的算<em>未找到</em>吗？换一个错误码会不会更诚实？
4. 在第 4 节的原始 stdio 单元格中，发送一个不带 `uri` 参数的 `resources/read`。返回的是什么错误码？它和资源缺失的情况有什么不同？